# Documents and Text Splitting

Retrieval-augmented pipelines do not embed entire PDFs or wikis as single vectors. They **chunk** text into smaller units, embed each chunk, and retrieve the most relevant pieces at query time. LangChain represents each chunk as a **`Document`**: a string **`page_content`** plus a **`metadata`** dict for filtering, citation, and traceability.

This notebook covers the **`Document`** model, why chunking matters, **`RecursiveCharacterTextSplitter`** and other splitters from **`langchain-text-splitters`**, chunk size and overlap trade-offs, metadata conventions, markdown and code-aware splitting, measuring chunk quality, and preparing a sample corpus for embedding in **09. Embeddings and Vector Stores**.

**07. Streaming, Batch, and Async** completed the Runnable execution modes. From here the track shifts toward **retrieval**: documents → embeddings → vector store → retriever → RAG (**09–11**).

In [ ]:
import json
from pathlib import Path

import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain-core:", langchain_core.__version__)

---

## 1. Why Documents and Chunks?

Embedding models accept **limited input length** (thousands of tokens, not millions). Vector search returns **top-k** results — if each result is an entire manual, the LLM context fills with irrelevant text. **Chunking** balances:

| Goal | Chunking helps by… |
|------|---------------------|
| **Retrieval precision** | Smaller units match specific queries |
| **Context fit** | Retrieved text fits in the LLM window |
| **Citation** | Metadata points to source page/section |
| **Update granularity** | Re-index one section, not the whole book |

```
Raw files (PDF, MD, HTML)
        │
        ▼
   Load / extract text
        │
        ▼
   split into Documents  ◄── this notebook
        │
        ▼
   embed + store         ◄── 09. Embeddings and Vector Stores
        │
        ▼
   retrieve top-k        ◄── 10. Retrievers
```

---

## 2. The Document Model

**`Document`** (`langchain_core.documents`) is LangChain's standard text carrier:

| Field | Type | Purpose |
|-------|------|--------|
| **`page_content`** | `str` | Text embedded and sent to the LLM |
| **`metadata`** | `dict` | Source path, page number, section, ACL tags |

Retrievers return **`list[Document]`**. RAG prompts format **`page_content`**; UIs display **`metadata`** for citations.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="JWT bearer tokens authenticate API requests via Authorization header.",
    metadata={"id": "c3", "source": "security-docs", "topic": "auth"},
)

print("content:", doc.page_content)
print("metadata:", doc.metadata)
print("type:", type(doc).__name__)

### 2.1 Metadata Conventions

Consistent metadata simplifies filtering and citations:

| Key | Example | Use |
|-----|---------|-----|
| **`source`** | `api-docs.md` | File or URL origin |
| **`id`** | `c3`, `chunk-0042` | Stable chunk identifier |
| **`page`** | `12` | PDF page number |
| **`section`** | `Authentication` | Heading / chapter |
| **`updated_at`** | ISO timestamp | Freshness filtering |

Keep metadata **JSON-serializable** (str, int, float, bool, lists of same).

In [ ]:
RAW_CORPUS = [
    {
        "id": "c1",
        "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE.",
        "source": "api-docs",
    },
    {
        "id": "c2",
        "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions.",
        "source": "db-docs",
    },
    {
        "id": "c3",
        "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header.",
        "source": "security-docs",
    },
    {
        "id": "c4",
        "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines.",
        "source": "test-docs",
    },
    {
        "id": "c5",
        "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files.",
        "source": "db-docs",
    },
]

documents = [
    Document(page_content=item["text"], metadata={"id": item["id"], "source": item["source"]})
    for item in RAW_CORPUS
]

print(f"{len(documents)} pre-chunked documents")
for d in documents:
    print(f"  [{d.metadata['id']}] {d.page_content[:55]}...")

---

## 3. Chunk Size and Overlap

Splitters take **`chunk_size`** (max characters or tokens) and **`chunk_overlap`** (shared characters between adjacent chunks).

| Parameter | Too small | Too large |
|-----------|-----------|-----------|
| **`chunk_size`** | Loses context; fragmented answers | Imprecise retrieval; wastes tokens |
| **`chunk_overlap`** | `0` — facts split across boundaries get lost | High overlap — duplicate storage and cost |

Starting points (tune per corpus):

| Corpus type | Typical chunk_size | overlap |
|-------------|-------------------|--------|
| Technical docs | 500–1000 chars | 50–150 |
| Chat logs | 300–600 | 30–80 |
| Legal / long prose | 1000–2000 | 100–200 |

Token-based splitters (**§6**) align better with embedding model limits than raw character counts.

---

## 4. RecursiveCharacterTextSplitter — Default Choice

**`RecursiveCharacterTextSplitter`** tries separators in order — `\n\n`, `\n`, space, then character — to keep paragraphs and sentences intact when possible. It is the **recommended default** for most text.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

LONG_TEXT = """# Notes API Overview

The Notes API is built with FastAPI and stores notes in memory for demos.
Routes live under /notes with standard CRUD handlers.

## Authentication

JWT bearer tokens authenticate API requests.
Clients send Authorization: Bearer <token> in the request header.

## Database Migrations

Alembic applies SQLAlchemy schema migrations.
Run alembic upgrade head after pulling new revisions.
Autogenerate compares models to the live database and drafts revision files.

## Testing

Pytest fixtures share database setup across tests.
Place session-scoped engines in conftest.py.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

text_chunks = splitter.split_text(LONG_TEXT)
print(f"split_text → {len(text_chunks)} chunks\n")
for i, c in enumerate(text_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)
    print()

### 4.1 split_documents — Preserve Metadata

**`split_documents`** copies parent metadata to each chunk and can add **`start_index`**:

In [ ]:
parent = Document(
    page_content=LONG_TEXT,
    metadata={"source": "notes-api-guide.md", "id": "guide-v1"},
)

splitter_meta = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    add_start_index=True,
)

child_docs = splitter_meta.split_documents([parent])
print(f"split_documents → {len(child_docs)} Document objects\n")
for d in child_docs[:3]:
    print("metadata:", d.metadata)
    print("content:", repr(d.page_content[:70]), "...\n")

### 4.2 Custom Separators

Override default separators for log files, CSV-like text, or custom delimiters:

In [ ]:
log_splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=0,
    separators=["\n---\n", "\n", " ", ""],
)

logs = "2024-01-01 INFO started\n---\n2024-01-01 ERROR timeout\n---\n2024-01-02 INFO recovered"
for i, c in enumerate(log_splitter.split_text(logs)):
    print(i, repr(c))

---

## 5. CharacterTextSplitter — Fixed Separator

**`CharacterTextSplitter`** splits on a **single** separator (default `\n\n`). Simpler but less adaptive than recursive splitting.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=10,
)

char_chunks = char_splitter.split_text(LONG_TEXT)
print(f"CharacterTextSplitter → {len(char_chunks)} chunks")
print("first chunk:", repr(char_chunks[0][:80]), "...")

---

## 6. TokenTextSplitter — Token-Aware Chunks

**`TokenTextSplitter`** uses **`tiktoken`** to split by **token count** — aligned with embedding and LLM context limits.

In [ ]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=40,
    chunk_overlap=8,
    encoding_name="cl100k_base",  # OpenAI embedding / chat family
)

token_chunks = token_splitter.split_text(LONG_TEXT)
print(f"TokenTextSplitter → {len(token_chunks)} chunks")
print("chunk 0 tokens (~):", len(token_chunks[0].split()))
print(token_chunks[0][:200], "...")

Use token splitters when your **`chunk_size`** must stay under embedding model limits (e.g. 8191 tokens for `text-embedding-3-small` input).

---

## 7. MarkdownHeaderTextSplitter — Structure-Aware

Split markdown by **headers** so chunks respect document outline. Headers propagate into metadata.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_docs = md_splitter.split_text(LONG_TEXT)

print(f"MarkdownHeaderTextSplitter → {len(md_docs)} sections\n")
for d in md_docs:
    print("metadata:", d.metadata)
    print("content:", d.page_content[:80].replace("\n", " "), "...\n")

**Two-stage pattern:** split by headers first, then apply **`RecursiveCharacterTextSplitter`** on oversized sections.

In [ ]:
md_sections = md_splitter.split_text(LONG_TEXT)
fine_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=15)
final_docs = fine_splitter.split_documents(md_sections)

print(f"Two-stage: {len(md_sections)} sections → {len(final_docs)} chunks")
print("sample metadata:", final_docs[2].metadata)

---

## 8. Language-Aware Code Splitting

**`RecursiveCharacterTextSplitter.from_language`** uses language-specific separators (Python: `def`, `class`, …).

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

PYTHON_SAMPLE = '''
def authenticate(token: str) -> bool:
    """Validate JWT bearer token."""
    return bool(token)

class NotesService:
    def list_notes(self):
        return []

    def create_note(self, title: str):
        return {"title": title}
'''

py_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=120,
    chunk_overlap=10,
)

code_doc = Document(page_content=PYTHON_SAMPLE, metadata={"source": "notes_service.py"})
code_chunks = py_splitter.split_documents([code_doc])

for i, d in enumerate(code_chunks):
    print(f"--- code chunk {i} ---")
    print(d.page_content)
    print()

---

## 9. Measuring Chunk Quality

Inspect distributions before embedding — avoid tiny fragments and oversized outliers.

In [ ]:
def chunk_stats(docs: list[Document]) -> dict:
    lengths = [len(d.page_content) for d in docs]
    return {
        "count": len(lengths),
        "min": min(lengths),
        "max": max(lengths),
        "mean": float(np.mean(lengths)),
        "median": float(np.median(lengths)),
    }


production_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
guide_doc = Document(page_content=LONG_TEXT, metadata={"source": "guide.md"})
prod_chunks = production_splitter.split_documents([guide_doc])

stats = chunk_stats(prod_chunks)
print(json.dumps(stats, indent=2))

lengths = [len(d.page_content) for d in prod_chunks]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=12, color="#4c72b0", edgecolor="white")
ax.axvline(production_splitter._chunk_size, color="red", linestyle="--", label="chunk_size limit")
ax.set_xlabel("characters per chunk")
ax.set_ylabel("count")
ax.set_title("Chunk length distribution")
ax.legend()
plt.tight_layout()
plt.show()

---

## 10. Enriching Chunk Metadata

After splitting, add **`chunk_index`**, **`parent_id`**, or prepend headings for retrieval context:

In [ ]:
def enrich_chunks(chunks: list[Document], parent_id: str) -> list[Document]:
    enriched = []
    for i, doc in enumerate(chunks):
        meta = {**doc.metadata, "parent_id": parent_id, "chunk_index": i}
        enriched.append(Document(page_content=doc.page_content, metadata=meta))
    return enriched


enriched = enrich_chunks(prod_chunks, parent_id="guide-v1")
print(enriched[0].metadata)
print(enriched[-1].metadata)

### 10.1 Contextual Headers in page_content

Some pipelines **prepend** section titles to each chunk so embeddings capture hierarchy:

In [ ]:
def prepend_heading(doc: Document) -> Document:
    h2 = doc.metadata.get("h2", "")
    prefix = f"Section: {h2}\n\n" if h2 else ""
    return Document(page_content=prefix + doc.page_content, metadata=doc.metadata)


contextual = [prepend_heading(d) for d in md_docs]
print(contextual[1].page_content[:120], "...")

---

## 11. format_document — Prompt-Ready Strings

LangChain helpers format **`Document`** objects for injection into prompts:

In [ ]:
from langchain_core.prompts import format_document

sample = Document(
    page_content="JWT bearer tokens use Authorization header.",
    metadata={"id": "c3", "source": "security-docs"},
)

formatted = format_document(sample, template="[{id}] {page_content}")
print(formatted)

RAG chains often use a custom **`format_docs`** lambda (**06**, **11**) — same idea with full control over layout.

---

## 12. Loading from Files (Pattern)

LangChain **`langchain-community`** loaders produce `Document` lists from PDFs, HTML, etc. This track uses explicit text for clarity; the ingest pattern is:

In [ ]:
def load_text_file(path: Path) -> Document:
    text = path.read_text(encoding="utf-8")
    return Document(page_content=text, metadata={"source": str(path.name)})


# Example: write temp file and load
demo_path = Path("_demo_notes_doc.txt")
demo_path.write_text(LONG_TEXT, encoding="utf-8")

loaded = load_text_file(demo_path)
loaded_chunks = production_splitter.split_documents([loaded])
print(f"Loaded {demo_path.name} → {len(loaded_chunks)} chunks")

demo_path.unlink(missing_ok=True)

---

## 13. Splitter Selection Guide

| Splitter | Best for |
|----------|----------|
| **`RecursiveCharacterTextSplitter`** | General text (default) |
| **`CharacterTextSplitter`** | Single-delimiter logs |
| **`TokenTextSplitter`** | Token budget alignment |
| **`MarkdownHeaderTextSplitter`** | Markdown / wiki docs |
| **`from_language(PYTHON, …)`** | Source code |
| **Two-stage** (headers + recursive) | Long structured docs |

---

## 14. Parent–Document Retriever (Concept)

Advanced RAG embeds **small chunks** for precision but returns **larger parent spans** for LLM context. LangChain supports parent-document retriever patterns in integration packages — conceptually:

```
small child chunks  →  embed & search
parent document id  →  fetch wider context for generation
```

Implementation details tie into **10. Retrievers** and **11. RAG with LCEL**. Metadata fields like **`parent_id`** (§10) enable this pattern.

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| `chunk_size` too large | Poor retrieval precision | Reduce size; measure hit rate |
| Zero overlap | Sentences split across chunks | Add 10–15% overlap |
| Losing metadata on split | No citations | Use `split_documents` |
| Embedding raw HTML tags | Noisy vectors | Strip markup before split |
| One chunk per file for huge PDFs | Never retrieve middle sections | Always split long docs |
| Character count vs token limit | Embedding API errors | Use `TokenTextSplitter` |
| Duplicate chunk ids | Wrong citations | Add `chunk_index` or UUID |

---

## 16. Build the Lesson Corpus

Consolidate pre-chunked **`documents`** (short notes) and split **`prod_chunks`** (long guide) for use in **09**:

In [ ]:
LESSON_CORPUS: list[Document] = documents + enriched

print(f"LESSON_CORPUS: {len(LESSON_CORPUS)} documents ready for embedding")
print("ids:", [d.metadata.get("id", d.metadata.get("chunk_index")) for d in LESSON_CORPUS[:8]], "...")

---

## 17. Summary

**Documents** carry **`page_content`** and **`metadata`**. **Text splitters** break large sources into retrieval-sized chunks — **`RecursiveCharacterTextSplitter`** is the default; **`TokenTextSplitter`** and **`MarkdownHeaderTextSplitter`** address token limits and structure.

Key takeaways:

- Chunk for **retrieval precision** and **context window** fit, not arbitrary file boundaries.
- Tune **`chunk_size`** and **`chunk_overlap`**; measure length distributions before embedding.
- Use **`split_documents`** to preserve and extend metadata.
- **Two-stage splitting** (headers → recursive) works well for markdown knowledge bases.
- Enrich chunks with **`chunk_index`**, **`parent_id`**, and optional heading prefixes.

Demonstrations covered Document creation, recursive/character/token/markdown/code splitters, chunk stats, metadata enrichment, `format_document`, and a consolidated **`LESSON_CORPUS`** for the next notebook.

Next: **09. Embeddings and Vector Stores** — `OpenAIEmbeddings`, Chroma, indexing, and similarity search.